# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86f788257a', 'id': 'chatcmpl-DuunIMykOJkmVfhRj1qDi1rf5J5op', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02b8-3a49-71a0-a09e-c6e64aa19ccc-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool

In [3]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열

    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [5]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [6]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [7]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iy1aacvc5snyuXkdZYfe2Oeh', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DuunJzDkFUyXGHun3irWsXzYTppzx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b8-3f7a-7402-919a-c9d1dc7e61c5-0', tool_cal

In [8]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_iy1aacvc5snyuXkdZYfe2Oeh',
  'type': 'tool_call'}]

In [9]:
tool_dict[response.tool_calls[0]['name']].invoke(response.tool_calls[0])

Asia/Seoul (서울) 현재시각 2026-06-26 15:57:38 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:57:38 ', name='get_current_time', tool_call_id='call_iy1aacvc5snyuXkdZYfe2Oeh')

In [10]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [11]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x0000024A3BD10E50>)

In [12]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (서울) 현재시각 2026-06-26 15:57:38 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:57:38 ', name='get_current_time', tool_call_id='call_iy1aacvc5snyuXkdZYfe2Oeh')

In [13]:
for i in [1,2,3,4]:
    print(i)

1
2
3
4


In [14]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_iy1aacvc5snyuXkdZYfe2Oeh',
  'type': 'tool_call'}]

In [15]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 15:57:39 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iy1aacvc5snyuXkdZYfe2Oeh', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DuunJzDkFUyXGHun3irWsXzYTppzx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b8-3f7a-7402-919a-c9d1dc7e61c5-0', tool_cal

In [16]:
llm_with_tools.invoke(messages)

AIMessage(content='현재 서울의 시각은 2026년 6월 26일 15시 57분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 189, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DuunMBg2q1kJGe3fMPwSnYgmwy7Zp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02b8-4ba0-7333-9b15-2100289f9d78-0', usage_metadata={'input_tokens': 189, 'output_tokens': 25, 'total_tokens': 214, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [17]:
messages.append(llm_with_tools.invoke(messages))

In [18]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iy1aacvc5snyuXkdZYfe2Oeh', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DuunJzDkFUyXGHun3irWsXzYTppzx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02b8-3f7a-7402-919a-c9d1dc7e61c5-0', tool_cal

### 주가 예제

In [28]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [29]:
messages = []
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [30]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_mmX05BOmBFGB9lgdnezp96kv',
  'type': 'tool_call'}]

In [31]:
messages

[HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_mmX05BOmBFGB9lgdnezp96kv', 'function': {'arguments': '{"ticker":"TSLA","period":"1mo"}', 'name': 'get_yf_stock_history'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 203, 'total_tokens': 226, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d8f5eee12', 'id': 'chatcmpl-DuusEYxTWk7e2OLNroScYiHSWEljw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02bc-e5f0-7c60-a95e-40f69e92f321-0', tool_calls=[{'name': 'get_yf_stock_history', 'args': {'ticker': 'TSLA', 'period': '1mo'}, 'id': 'call_mmX05B

In [32]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습

for tool_call in response.tool_calls:
   out = tool_dict[tool_call['name']].invoke(tool_call)
   messages.append(out)

llm_with_tools.invoke(messages)


AIMessage(content='한달 전 테슬라(TSLA)의 주가는 약 430.26달러로 시작했으며, 현재 주가는 약 375.12달러입니다. \n\n이는 주가가 한달 사이에 내렸음을 의미합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 1559, 'total_tokens': 1611, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d8f5eee12', 'id': 'chatcmpl-Duusfieb0N2ZAbOs7XlKs7Wpp1Ouv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02bd-50d6-7ec3-8766-e29959787e20-0', usage_metadata={'input_tokens': 1559, 'output_tokens': 52, 'total_tokens': 1611, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [1]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [2]:
from pdf_summary import summarize_pdf

In [3]:
tool_list = [summarize_pdf]
tool_dict = {'summarize_pdf':summarize_pdf}

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model ='gpt-4o-mini')

In [6]:
llm_with_tools = llm.bind_tools(tool_list)

In [8]:
response = llm_with_tools.invoke(messages)

In [11]:
messages.append(response)

In [12]:
for tool_call in response.tool_calls:
    out = tool_dict[tool_call['name']].invoke(tool_call)

In [14]:
messages.append(out)

In [15]:
llm_with_tools.invoke(messages)

AIMessage(content='문서의 저자는 Qianzi Yu, Kai Zhu, Yang Cao, Yu Kang입니다. 이들은 중국 과학기술대학교 및 헵페이 종합 국가 과학 센터의 연구원들로, 인공지능 및 비디오 분석 분야에서 활발히 연구하고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 600, 'total_tokens': 663, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_88876bec1e', 'id': 'chatcmpl-DuvNFGeqDYI4ezG8XcKmRIulKKkbK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02da-3f31-75b2-8ce7-f9d39a632c35-0', usage_metadata={'input_tokens': 600, 'output_tokens': 63, 'total_tokens': 663, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})